#  Decision Tree Classification — Pipeline
### Titanic Survival Prediction with sklearn Pipeline

---

##  Overview

This notebook demonstrates a **production-ready ML pipeline** using scikit-learn's `Pipeline` and `ColumnTransformer` to predict Titanic passenger survival.

A **Pipeline** chains multiple preprocessing steps and a final estimator into a single object. This ensures:
-  No data leakage (transformers are fit only on training data)
-  Clean, reproducible, deployable code
-  Easy hyperparameter tuning with `GridSearchCV`

**Pipeline Steps:**

| Step | Transformer | Purpose |
|------|-------------|----------|
| 1 | `SimpleImputer` | Fill missing values |
| 2 | `OneHotEncoder` | Encode categorical columns |
| 3 | `MinMaxScaler` | Scale features to [0, 1] range |
| 4 | `SelectKBest` | Select top K most informative features |
| 5 | `DecisionTreeClassifier` | Final classification model |

---

##  About the Titanic Dataset

The **Titanic dataset** contains information about 891 passengers. The goal is to predict `Survived` (0 = No, 1 = Yes).

**Features Used:**
- `Pclass` — Ticket class (1st, 2nd, 3rd)
- `Sex` — Gender (categorical)
- `Age` — Age in years (has missing values)
- `SibSp` — # siblings/spouses aboard
- `Parch` — # parents/children aboard
- `Fare` — Passenger fare
- `Embarked` — Port of embarkation (categorical, has missing values)

### Import Required Libraries

In [56]:
import pandas as pd                                              # Data manipulation
import seaborn as sns                                           # (available for plotting if needed)
from sklearn.model_selection import train_test_split            # Train/test splitting
from sklearn.impute import SimpleImputer                        # Fill missing values
from sklearn.preprocessing import OneHotEncoder                 # Encode categorical features
from sklearn.preprocessing import MinMaxScaler, StandardScaler  # Feature scaling
from sklearn.tree import DecisionTreeClassifier                 # Decision Tree model
from sklearn.compose import ColumnTransformer                   # Apply transformers to specific columns
from sklearn.pipeline import make_pipeline                      # Chain steps into a pipeline
from sklearn.feature_selection import SelectKBest, chi2         # Feature selection using chi-squared test
from sklearn.metrics import accuracy_score                      # Evaluation metric
from sklearn.model_selection import cross_val_score             # K-Fold cross-validation
from sklearn.model_selection import GridSearchCV                # Hyperparameter tuning
from sklearn.metrics import classification_report

### Load Data and Prepare Features

In [58]:
# Load Titanic training data
data = pd.read_csv("train.csv")

# Drop columns that are not useful as features:
# - PassengerId: just a row identifier
# - Name: text with too many unique values
# - Ticket: alphanumeric codes with no clear pattern
# - Cabin: too many missing values (~77%)
# - Survived: this is our TARGET variable, not a feature
X = data.drop(['PassengerId', 'Name', 'Ticket', 'Cabin', 'Survived'], axis=1)
X.head(2)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,male,22.0,1,0,7.2500,S
1,1,female,38.0,1,0,71.2833,C


In [59]:
# Target variable: 0 = Did not survive, 1 = Survived
y = data['Survived']
y.head(2)

0    0
1    1
Name: Survived, dtype: int64

### Train / Test Split

> **Important:** The split is done **before** any preprocessing. The pipeline will handle all transformations internally — and crucially, it will fit transformers **only on training data** to prevent leakage.

In [61]:
# 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print('Shape of X_train : ', X_train.shape)
print('Shape of y_train : ', y_train.shape)
print('Shape of X_test  : ', X_test.shape)
print('Shape of y_test  : ', y_test.shape)

Shape of X_train :  (712, 7)
Shape of y_train :  (712,)
Shape of X_test  :  (179, 7)
Shape of y_test  :  (179,)


### Build the Pipeline

Each step is a `ColumnTransformer` that applies a specific transformation to specific columns.

**Column Index Reference** (after dropping unused columns):

| Index | Column |
|-------|--------|
| 0 | Pclass |
| 1 | Sex |
| 2 | Age |
| 3 | SibSp |
| 4 | Parch |
| 5 | Fare |
| 6 | Embarked |

In [63]:
# ──────────────────────────────────────────
# STEP 1: Imputation — Fill Missing Values
# ──────────────────────────────────────────
# Age (col 2): fill missing values with the column MEAN (default strategy)
# Embarked (col 6): fill missing values with the MOST FREQUENT value ('S')
# remainder='passthrough' → all other columns pass through unchanged
clf1 = ColumnTransformer([
    ('impute_age',      SimpleImputer(),                          [2]),   # Mean imputation for Age
    ('impute_embarked', SimpleImputer(strategy='most_frequent'),  [6])    # Mode imputation for Embarked
], remainder='passthrough')
clf1

ColumnTransformer(remainder='passthrough',
                  transformers=[('impute_age', SimpleImputer(), [2]),
                                ('impute_embarked',
                                 SimpleImputer(strategy='most_frequent'),
                                 [6])])

In [64]:
# ──────────────────────────────────────────────────────
# STEP 2: One-Hot Encoding — Convert Categorical to Numeric
# ──────────────────────────────────────────────────────
# Sex (col 1) and Embarked (col 6) are categorical strings
# OHE converts each category into a binary column
# sparse_output=False → return dense array (not sparse matrix)
# handle_unknown='ignore' → silently ignore unseen categories at test time
clf2 = ColumnTransformer([
    ('ohe_sex_embarked', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), [1, 6])
], remainder='passthrough')
clf2

ColumnTransformer(remainder='passthrough',
                  transformers=[('ohe_sex_embarked',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 [1, 6])])

In [65]:
# ──────────────────────────────────────────────
# STEP 3: Feature Scaling — MinMaxScaler
# ──────────────────────────────────────────────
# MinMaxScaler scales all features to the range [0, 1]: x_scaled = (x - min) / (max - min)
# slice(0,10) applies scaling to the first 10 columns after OHE expansion
# 
# requires NON-NEGATIVE input — StandardScaler can produce negative values!
# MinMaxScaler always produces values in [0,1] which satisfies chi2's requirement.
clf3 = ColumnTransformer([
    ('scale', MinMaxScaler(), slice(0, 10))   # Scale all feature columns
])
clf3

ColumnTransformer(transformers=[('scale', MinMaxScaler(), slice(0, 10, None))])

In [66]:
# ─────────────────────────────────────────────────────────
# STEP 4: Feature Selection — SelectKBest with Chi-Squared
# ─────────────────────────────────────────────────────────
# SelectKBest keeps only the K features most correlated with the target
# chi2 (chi-squared test) measures statistical dependence between each feature and y
# k=8 → keep the 8 most informative features out of all available
# 
clf4 = SelectKBest(score_func=chi2, k=8)
clf4

SelectKBest(k=8, score_func=<function chi2 at 0x000001BEF8C923E0>)

In [67]:
# ────────────────────────────────────────────────────
# STEP 5: Decision Tree Classifier (Final Estimator)
# ────────────────────────────────────────────────────
# max_depth=2 → very shallow tree, reduces overfitting
# This is intentionally simple to start — GridSearchCV will tune it later
clf5 = DecisionTreeClassifier(max_depth=2)
clf5

DecisionTreeClassifier(max_depth=2)

### Assemble the Full Pipeline

`make_pipeline()` chains all steps in order. When you call `pipe_line.fit(X_train, y_train)`, it:
1. Fits & transforms through steps 1–4 on training data
2. Passes the result to `clf5.fit()` for training

When you call `pipe_line.predict(X_test)`, it:
1. Only **transforms** (doesn't re-fit) through steps 1–4
2. Passes the result to `clf5.predict()`

In [69]:
# Chain all 5 steps into a single pipeline object
pipe_line = make_pipeline(clf1, clf2, clf3, clf4, clf5)
pipe_line

Pipeline(steps=[('columntransformer-1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('columntransformer-2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_sex_embarked',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 6])])),
                ('columntransformer-3',
                 ColumnTransformer(transformers=[('scale', MinMaxScaler(),
                                                  slice(0, 10, None))])),
                ('selectkbest',
                 SelectKBest(k=8,
                             score_func=<function chi2 at 0x000001BEF8C923E0>)),
                ('decisiontreeclassifier',
                 DecisionTreeClassifier(max_depth=2))])

### Fit the Pipeline on Training Data

In [71]:
# Fit the entire pipeline on training data
# Each step is automatically fitted in sequence — no manual transform calls needed
pipe_line.fit(X_train, y_train)

Pipeline(steps=[('columntransformer-1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('columntransformer-2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_sex_embarked',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 6])])),
                ('columntransformer-3',
                 ColumnTransformer(transformers=[('scale', MinMaxScaler(),
                                                  slice(0, 10, None))])),
                ('selectkbest',
                 SelectKBest(k=8,
                             score_func=<function chi2 at 0x000001BEF8C923E0>)),
                ('decisiontreeclassifier',
                 DecisionTreeClassifier(max_depth=2))])

### Make Predictions

In [73]:
# Run test data through the full pipeline (transform only, no re-fit)
y_pred = pipe_line.predict(X_test)
y_pred

array([1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1,
       0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1,
       0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1,
       0, 0, 0], dtype=int64)

In [74]:
# Evaluate: proportion of correctly predicted survival outcomes
print("Accuracy Score For Decision Tree [Classification] ::", accuracy_score(y_test, y_pred))

Accuracy Score For Decision Tree [Classification] :: 0.6256983240223464


### Cross-Validation

**Cross-Validation** gives a more reliable estimate of model performance than a single train/test split.

`cv=5` means **5-Fold Cross-Validation**:
- Training data is split into 5 equal folds
- Model trains on 4 folds, validates on 1 fold — repeated 5 times
- Final score = mean across all 5 validation scores

> **Why use CV?** A single train/test split can be lucky or unlucky. CV gives a more stable and reliable performance estimate.

In [76]:
# 5-Fold Cross-Validation on TRAINING data only
# This gives a reliable estimate of how well the pipeline generalizes
cv_scores = cross_val_score(pipe_line, X_train, y_train, cv=5, scoring='accuracy')

print(f"CV Scores per Fold : {cv_scores}")
print(f"Mean CV Score      : {cv_scores.mean():.4f}")
print(f"Std of CV Scores   : {cv_scores.std():.4f}")   # Low std = stable model

CV Scores per Fold : [0.6013986  0.62237762 0.68309859 0.65492958 0.63380282]
Mean CV Score      : 0.6391
Std of CV Scores   : 0.0280


### Hyperparameter Tuning with GridSearchCV

**GridSearchCV** exhaustively searches over a grid of hyperparameter values and returns the best combination.

**Naming convention for pipeline parameters:**
`stepname__parameter` (double underscore)

Since `make_pipeline` auto-names steps by class name (lowercase), the Decision Tree step is named `decisiontreeclassifier`, so `max_depth` is tuned as `decisiontreeclassifier__max_depth`.

In [78]:
# Define the parameter grid to search
# Key format: 'stepname__param_name': [list of values to try]
param_grid = {
    'decisiontreeclassifier__max_depth': [1, 2, 3, 4, 5, None]   # Try 6 different depths
}

# GridSearchCV tries all combinations with 5-fold CV
grid = GridSearchCV(
    pipe_line,          # The full pipeline
    param_grid,         # Parameters to search
    cv=5,               # 5-fold cross-validation for each candidate
    scoring='accuracy'  # Optimize for accuracy
)

# Fit GridSearchCV — this trains the pipeline 5 × 6 = 30 times
grid.fit(X_train, y_train)

print(f"GridSearch Best CV Score  : {grid.best_score_:.4f}")
print(f"GridSearch Best Parameters: {grid.best_params_}")

GridSearch Best CV Score  : 0.6391
GridSearch Best Parameters: {'decisiontreeclassifier__max_depth': 2}


### Use the best model from GridSearch for final predictions

In [80]:
best_model = grid.best_estimator_
y_pred_best = best_model.predict(X_test)
print(accuracy_score(y_test, y_pred_best))

0.6256983240223464


### Classification Report 

In [82]:
print(classification_report(y_test,y_pred_best))

              precision    recall  f1-score   support

           0       0.64      0.83      0.72       105
           1       0.58      0.34      0.43        74

    accuracy                           0.63       179
   macro avg       0.61      0.58      0.57       179
weighted avg       0.62      0.63      0.60       179



### Key Takeaways

1. **Pipelines prevent data leakage** — All transformers are fit only on training data. Test data is only transformed, never fit.

2. **ColumnTransformer** applies different preprocessing to different columns in a single step.

3. **MinMaxScaler over StandardScaler** — chi2 feature selection requires non-negative values. MinMaxScaler scales to [0,1]; StandardScaler can produce negatives.

4. **Cross-Validation** gives a more reliable performance estimate than a single train/test split.

5. **GridSearchCV with Pipeline** — Use `stepname__param` syntax to tune any step's hyperparameters.
 